# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`

This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata_dict = dataset.metadata.to_json()

print(f"Dataset name: {metadata_dict.get('name','')}")
print(f"Description: {metadata_dict.get('description','')}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We list available record sets and their structure using their Croissant `@id` fields.

In [ ]:
# List all record sets available in the dataset
print("Available record sets (`@id`):")
for record_set in dataset.metadata.record_sets:
    print(f"- @id: {record_set.id}")
    if hasattr(record_set, 'fields'):
        print("  Fields:")
        for field in record_set.fields:
            name_str = getattr(field, 'name', None) or getattr(field, 'description', None) or field.id
            print(f"    - {field.id} (name: {name_str})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# Collect record set @ids
record_set_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Record set: {record_set_id} - DataFrame shape: {df.shape}")

# Pick the first available record set for further analysis (customize if needed)
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"\nColumns in main record set ({main_record_set_id}):\n")
    print(dataframes[main_record_set_id].columns.tolist())
    display(dataframes[main_record_set_id].head())
else:
    main_record_set_id = None
    print("No record sets found.")

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
import numpy as np

if main_record_set_id:
    df = dataframes[main_record_set_id]
    # Print numeric columns for selection
    numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
    print(f"Numeric columns available: {numeric_cols}")
    
    if numeric_cols:
        # Pick the first numeric field for demonstration (customize as needed)
        numeric_field = numeric_cols[0]
        print(f"Example numeric field selected: {numeric_field}")
        
        threshold = df[numeric_field].mean() if df[numeric_field].notnull().sum()>0 else 0
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records in '{main_record_set_id}' with {numeric_field} > {threshold:.2f} (mean):")
        display(filtered_df.head())
        
        norm_col = f"{numeric_field}_normalized"
        filtered_df[norm_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, norm_col]].head())

        # Grouping by a suitable group field if found
        # Look for a string/object or low-cardinality column
        object_cols = [c for c in df.columns if df[c].dtype == object or df[c].dtype.name == 'category']
        group_field = None
        for col in object_cols:
            if 1 < df[col].nunique() < 20:
                group_field = col
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().to_frame()
            print(f"Grouped mean of {numeric_field} by '{group_field}':")
            display(grouped_df)
        else:
            print("No suitable group field found for grouping example.")
    else:
        print("No numeric fields found for analysis in the selected record set.")
else:
    print("Skipping EDA: No record set available.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_cols:
    plt.figure(figsize=(8,5))
    sns.histplot(dataframes[main_record_set_id][numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of '{numeric_field}'")
    plt.xlabel(numeric_field)
    plt.ylabel('Count')
    plt.show()
    
    # If group_field found, visualize group means
    if 'group_field' in locals() and group_field:
        group_means = dataframes[main_record_set_id].groupby(group_field)[numeric_field].mean().reset_index()
        plt.figure(figsize=(10,5))
        sns.barplot(x=group_field, y=numeric_field, data=group_means)
        plt.title(f"Mean {numeric_field} by {group_field}")
        plt.show()

## 6. Conclusion
In this notebook, we loaded and explored the FAIR^2 dataset
(*Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells*) using the mlcroissant library.

- We programmatically listed and referenced all dataset structures by their Croissant `@id`s.
- We performed basic data extraction, checked available numeric fields, filtered and normalized data, and visualized feature distributions.
- This workflow can be adapted for further analysis, e.g., ML pipelines, by referencing the appropriate record set and field `@id`s as needed.